## Cell 1: Start PySpark with Extra Memory

In [1]:
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] += ";" + "C:\\hadoop\\bin"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Give Spark 8GB of memory to handle the 2GB+ CSV file safely
spark = SparkSession.builder \
    .appName("Drought_Deep_Learning_Prep") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("Spark is ready!")

Spark is ready!


## Cell 2: Load the Data

We will load both the changing weather data (`train`) and the permanent land data (`soil`). PySpark will automatically figure out if the columns are numbers or text (`inferSchema=True`).

In [3]:
soil_df = spark.read.csv("file:///D:/ITC/Year 4/Semester II/Parallel/TP/Final Project/dataset/soil_data.csv", header=True, inferSchema=True)

train_df = spark.read.csv("file:///D:/ITC/Year 4/Semester II/Parallel/TP/Final Project/dataset/train_timeseries/train_timeseries.csv", header=True, inferSchema=True)

print("Loaded!")
print(soil_df.count())
print(train_df.count())

Loaded!
3109
19300680


## Cell 3: Join the Data Together

We combine them using the `fips` column (the county ID). We use a "left" join to keep all our daily weather rows and just attach the matching soil data to each one.

In [4]:
# Join the datasets
joined_df = train_df.join(soil_df, on="fips", how="left")

# Show the first 5 rows to make sure it worked
joined_df.show(5)

+----+----------+-------+------+-----+-----+------+------+-------+-------+---------+-----+-----+---------+---------+-----------+-----+---------+---------+-----------+-----+---------+---------+---------+------+------+------+------+------+------+------+------+-------+-------+-------+-------+-------------+-----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+---+---+---+---+---+---+---+
|fips|      date|PRECTOT|    PS| QV2M|  T2M|T2MDEW|T2MWET|T2M_MAX|T2M_MIN|T2M_RANGE|   TS|WS10M|WS10M_MAX|WS10M_MIN|WS10M_RANGE|WS50M|WS50M_MAX|WS50M_MIN|WS50M_RANGE|score|      lat|      lon|elevation|slope1|slope2|slope3|slope4|slope5|slope6|slope7|slope8|aspectN|aspectE|aspectS|aspectW|aspectUnknown|         WAT_LAND|        NVG_LAND|        URB_LAND|        GRS_LAND|        FOR_LAND|     CULTRF_LAND|     CULTIR_LAND|       CULT_LAND|SQ1|SQ2|SQ3|SQ4|SQ5|SQ6|SQ7|
+----+----------+-------+------+-----+-----+------+------+--

## Cell 4: Clean and Sort

Time-series deep learning needs data in the correct order. We also need to drop rows where the drought score is missing, because a model cannot train if it does not have the answer.

In [5]:
# 1. Convert the string 'date' column into a real Date type
cleaned_df = joined_df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

# 2. Drop any rows where the target 'score' is missing
cleaned_df = cleaned_df.dropna(subset=["score"])

# 3. Sort by County (fips) first, then by Date
cleaned_df = cleaned_df.orderBy("fips", "date")

print("Data cleaned and sorted!")

Data cleaned and sorted!


## Cell 5: Scale the Features

Deep learning models learn best when all numbers are small and on the same scale. We will pack all the feature columns into one vector, and then scale them.

In [6]:
# List the columns you want your model to learn from 
# (You can add or remove columns based on what you want to test)
# The Final Optimized Feature Roster
feature_columns = [
    'PS', 'elevation', 'T2M_MAX', 'QV2M', 
    'WS10M_RANGE', 'NVG_LAND', 'CULTRF_LAND', 'CULTIR_LAND', 
    'lat', 'lon', 'GRS_LAND', 'FOR_LAND', 'CULT_LAND', 
    'T2MDEW', 'T2MWET', 'T2M_RANGE', 'TS', 'WS50M_RANGE'
] # Add the rest of your columns here

# Fill any missing feature values with 0 so the model doesn't break
cleaned_df = cleaned_df.fillna(0, subset=feature_columns)

# Step 1: Pack them into a single column called 'unscaled_features'
assembler = VectorAssembler(inputCols=feature_columns, outputCol="unscaled_features")
assembled_df = assembler.transform(cleaned_df)

# Step 2: Scale the features
scaler = StandardScaler(inputCol="unscaled_features", outputCol="features", withStd=True, withMean=True)
scaler_model = scaler.fit(assembled_df)
final_df = scaler_model.transform(assembled_df)

# Look at the final features and the target score
final_df.select("fips", "date", "features", "score").show(5)

+----+----------+--------------------+-----+
|fips|      date|            features|score|
+----+----------+--------------------+-----+
|1001|2000-01-04|[0.66902425139515...|  1.0|
|1001|2000-01-11|[0.68922739102646...|  2.0|
|1001|2000-01-18|[0.68739074196906...|  2.0|
|1001|2000-01-25|[0.63596456836209...|  2.0|
|1001|2000-02-01|[0.79942633446997...|  1.0|
+----+----------+--------------------+-----+
only showing top 5 rows



In [8]:
import pickle
import numpy as np

# Extract the raw numbers from PySpark scaler
scaler_params = {
    "mean": np.array(scaler_model.mean),
    "std":  np.array(scaler_model.std)
}

# Save as plain Python dict — pickle can handle this fine
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler_params, f)

print("Scaler saved!")
print("Mean shape:", scaler_params["mean"].shape)
print("Std shape:",  scaler_params["std"].shape)

Scaler saved!
Mean shape: (18,)
Std shape: (18,)


## Cell 6: Save for Deep Learning

PySpark is done! Now we save this clean data as "Parquet" files. Parquet is much faster than CSV. Your deep learning framework (like PyTorch or TensorFlow) can load Parquet files extremely fast in batches.

In [9]:
# Save the final cleaned and scaled data
# It will create a folder called 'deep_learning_data.parquet'
final_df.select("fips", "date", "features", "score") \
    .write \
    .mode("overwrite") \
    .parquet("deep_learning_data.parquet")

print("Data successfully saved for Deep Learning!")

Data successfully saved for Deep Learning!
